# Sprint 3 — Fine-tune the head on YOUR data (keep Gustking backbone)

Strategy: **do NOT swap the model**. Fine-tune the head on our own volunteer+clone data (one TTS engine). **Leave-one-speaker-out** validation guarantees the held-out speaker's real+cloned clips are never in training — so the numbers are about real-vs-synthetic generalization, not speaker ID leakage.

Compute budget: LOSO folds train **only the head** (fast); the single final deployable model uses the fuller recipe. Run cells 1→2→3→4.

In [ ]:
# @title 1. Mount Drive + clone repo + hydrate dataset (same as sprint0/1/2)
from google.colab import drive
from pathlib import Path
import sys, os, pathlib, subprocess, random, shutil

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"
REPO_DIR = Path("/content/VoxDetect")

ML_BASE        = Path("/content/drive/MyDrive/VoxDetect/ml-core")
DATASET_DIR    = ML_BASE / "dataset"
RESULTS_DIR    = ML_BASE / "results"
CHECKPOINT_DIR = ML_BASE / "checkpoints"
results_dir = RESULTS_DIR
for d in (DATASET_DIR, RESULTS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_DIR = Path("/content/VoxDetect_data")

if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))

subprocess.run([
    sys.executable, str(REPO_DIR / "ml-core" / "scripts" / "organize_dataset.py"),
    "--raw-dir", str(DATASET_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
])
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
test_data_dir = LOCAL_DATA_DIR

!pip install -q torch torchaudio librosa soundfile transformers resemblyzer huggingface_hub numpy scipy
print("deps installed")

In [ ]:
# @title 2. Verify the folder depth for the speaker-partition guarantee
# SPEAKER must be the folder DIRECTLY above each clip (real/<lang>/<speaker>/clip.wav).
# If it isn't, finetune_head.py will hard-fail loudly (it asserts the exact depth)
# rather than silently sorting clips by the wrong folder.
import glob
real_clips = sorted(glob.glob(str(test_data_dir / "real/**/*.wav"), recursive=True))
cloned_clips = sorted(glob.glob(str(test_data_dir / "cloned/**/*.wav"), recursive=True))
print("real clips:", len(real_clips), " cloned clips:", len(cloned_clips))

# Show the actual relative path (indent) so we can SEE which level is the speaker.
print("\nSample real paths (one level deeper == different folder layout):")
for p in real_clips[:6]:
    print("   ", pathlib.Path(p).relative_to(test_data_dir))
print("\nSample cloned paths:")
for p in cloned_clips[:6]:
    print("   ", pathlib.Path(p).relative_to(test_data_dir))

# Detect speaker names (the folder DIRECTLY above each clip) and show counts.
def speakers_of(label):
    d = {}
    for x in glob.glob(str(test_data_dir / f"{label}/**/*.wav"), recursive=True):
        parent = str(pathlib.Path(x).parent)          # glob gives str -> wrap in Path
        d[parent] = d.get(parent, 0) + 1
    return d
real_sp, cloned_sp = speakers_of("real"), speakers_of("cloned")
print("\nSpeaker folders found (must be the clip's parent dir):")
for root, cnt in sorted(real_sp.items(), key=lambda kv: -kv[1]):
    rel = pathlib.Path(root).relative_to(test_data_dir)
    print(f"   real   {rel}  ({cnt} clips)")


# Pick one holdout speaker for the A/B comparison (the LAST real speaker by name).
import pathlib as _p
holdout = _p.Path(max(real_sp, key=lambda k: _p.Path(k).name)).name
print("\n[holdout] using speaker:", holdout)

# Build a real/ + cloned/ dir containing ONLY that speaker, for evaluate.py A/B.
def build_holdout_dir(sp):
    out = pathlib.Path("/content/VoxDetect_holdout")
    if out.exists(): shutil.rmtree(out)
    for i, label in enumerate(("real", "cloned")):
        idx = 0
        for p in glob.glob(str(test_data_dir / f"{label}/**/*.wav"), recursive=True):
            p = pathlib.Path(p)
            if p.parent.name == sp:          # speaker is the clip's parent dir
                lang = p.parent.parent.name  # real/<lang>/<speaker>/
                dest = out / label / f"{lang}_{idx:03d}.wav"
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dest)
                idx += 1
    return out
holdout_dir = build_holdout_dir(holdout)
print("holdout eval dir:", holdout_dir, "(real+cloned, speaker only)")

In [ ]:
# @title 3. LOSO cross-validation + final fine-tuned model (dual freeze schedules)
# LOSO folds train ONLY the head (fast, generalization signal). The FINAL model (the one
# that ships) is trained on all non-holdout speakers with the fuller recipe.
!python3 {REPO_DIR}/ml-core/scripts/finetune_head.py \
    --data-dir {test_data_dir} \
    --out-dir {CHECKPOINT_DIR}/ft_head_v1 \
    --epochs 5 --batch-size 4 --lr 1e-5 --augment \
    --holdout {holdout} \
    --results-csv {results_dir}/ablation_results.csv
print("\nDone. LOSO aggregates + holdout eval saved to", CHECKPOINT_DIR / "ft_head_v1" / "finetune_report.json")

In [ ]:
# @title 4. Cleanup — purge crash-recovery checkpoints (keep final model + report)
# Training wrote MANY 1.26GB per-epoch shards + a doubled checkpoints/ dir (multi-GB).
# The live_demo only needs the FINAL model (config.json + model.safetensors +
# preprocessor_config.json) + finetune_report.json. Delete the throwaway checkpoints/.
import shutil, glob, pathlib
ft_head = pathlib.Path(CHECKPOINT_DIR) / "ft_head_v1"
for ck in [ft_head / "checkpoints"]:
    if ck.exists():
        size_mb = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e6
        shutil.rmtree(ck)
        print(f"[cleanup] deleted {ck}  ({size_mb:.0f} MB)")
    else:
        print(f"[cleanup] no {ck} to delete")
print("\n[cleanup] final model intact:", sorted(p.name for p in ft_head.iterdir()))


In [ ]:
# @title 5. A/B: base ASVspoof vs fine-tuned, on the SAME unseen speaker
# TWO SEPARATE CLAIMS — do NOT conflate these on the slide:
#   * LOSO            = N-fold CV over ALL speakers (mean +/- std), train-only-the-head.
#   * TRUE HOLDOUT    = 1 speaker kept ENTIRELY out of the final model, single run.
# Below: run the PRETRAINED (no --checkpoint) model on the true-holdout speaker's clips,
# compare with the fine-tuned holdout numbers from finetune_report.json => measured lift.
import subprocess, sys, json, glob

out = str(results_dir / "sprint3_base_asvspoof_holdout.json")
r = subprocess.run([
    sys.executable, "-m", "evaluate",
    "--root", str(holdout_dir), "--variant", "wav2vec2",
    "--out", out, "--find-threshold",
], cwd=str(SRC_PKG))
print("base ASVspoof eval ->", out)

ft = json.loads(pathlib.Path(CHECKPOINT_DIR / "ft_head_v1" / "finetune_report.json").read_text())
base = json.loads(pathlib.Path(out).read_text()) if r.returncode == 0 and pathlib.Path(out).exists() else None

print("\n=================== CLAIM 2: TRUE HOLD-OUT A/B (1 unseen speaker)"
      " ===================")
print(f"  model: base ASVspoof : acc={base.get('accuracy')} fpr={base.get('fpr')} fnr={base.get('fnr')}" if base else "  base ASVspoof: <no JSON — check evaluate.py output>")
if isinstance(ft.get("holdout"), dict):
    h = ft["holdout"]
    print(f"  model: fine-tuned    : acc={h['finetuned_eval']['acc']:.3f} fpr={h['finetuned_eval']['fpr']:.3f} fnr={h['finetuned_eval']['fnr']:.3f}")
else:
    print("  fine-tuned    : <no holdout in report — did you pass --holdout?>")

print("\n=================== CLAIM 1: LOSO aggregate (N-fold CV, all speakers)"
      " ===================")
l = ft.get("loso") or {}
if l:
    for k in ("acc", "fpr", "fnr"):
        print(f"    {k}: {l[k]['mean']:.3f} +/- {l[k]['std']:.3f}   per-fold={[round(v,3) for v in l[k]['per_fold']]}")
print("\nNOTE: the holdout speaker above was ALSO one of the LOSO folds. Keep these")
print("two claims separate on the slide. Copy numbers into results.md.")